# Cloud Masking PlanetScope Imagery

This notebook shows how to generate cloud masks from PlanetScope scenes using OmniCloudMask.

Two workflows are covered:
- **Single scene** — load one scene and get a numpy mask directly with `predict_from_array`
- **Batch processing** — process many scenes and save masks as GeoTIFFs with `predict_from_load_func`

PlanetScope 4-band scenes have the band order: 1=Blue, 2=Green, 3=Red, 4=NIR.

In [ ]:
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from omnicloudmask import load_multiband, predict_from_array, predict_from_load_func

## 1. Single scene

Use `load_multiband` to read the Red (3), Green (2), and NIR (4) bands from a PlanetScope GeoTIFF, resampled to 10 m. Band indices follow rasterio convention (1-based).

In [ ]:
path_to_ps_tif = Path("/example/path/20171229_033105_0f1a_3B_AnalyticMS_SR.tif")

# PlanetScope 4-band order: 1=Blue, 2=Green, 3=Red, 4=NIR
# OmniCloudMask needs Red, Green, NIR
ps_band_order = [3, 2, 4]

In [ ]:
scene, profile = load_multiband(
    input_path=path_to_ps_tif, resample_res=10, band_order=ps_band_order
)
print(f"Scene shape: {scene.shape}")

In [ ]:
mask = predict_from_array(input_array=scene)
print(f"Mask shape: {mask.shape}")
print("Classes — 0: clear, 1: thick cloud, 2: thin cloud, 3: cloud shadow")

Visualise the result. Blue (band 1) is loaded separately for a true-colour display.

In [ ]:
# Load RGB for display (Red=3, Green=2, Blue=1)
rgb, _ = load_multiband(
    input_path=path_to_ps_tif, resample_res=10, band_order=[3, 2, 1]
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

rgb_display = np.clip(rgb / 3000, 0, 1).transpose(1, 2, 0)

axes[0].imshow(rgb_display)
axes[0].set_title("True Colour")
axes[0].axis("off")

axes[1].imshow(mask.squeeze(), cmap="tab10", vmin=0, vmax=3)
axes[1].set_title("OmniCloudMask (0=clear  1=thick  2=thin  3=shadow)")
axes[1].axis("off")

tab10 = plt.get_cmap("tab10")
mask_sq = mask.squeeze()
rgba = np.zeros((*mask_sq.shape, 4), dtype=float)
for cls in range(4):
    colour = tab10(cls / 9)
    where = mask_sq == cls
    rgba[where, :3] = colour[:3]
    rgba[where, 3] = 0.0 if cls == 0 else 0.4

axes[2].imshow(rgb_display)
axes[2].imshow(rgba)
axes[2].set_title("Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Zoom into the top-left quarter
h, w = mask_sq.shape
rs, re, cs, ce = 0, h // 2, 0, w // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(rgb_display[rs:re, cs:ce])
axes[0].set_title("True Colour (top-left)")
axes[0].axis("off")
axes[1].imshow(mask_sq[rs:re, cs:ce], cmap="tab10", vmin=0, vmax=3)
axes[1].set_title("OmniCloudMask (0=clear  1=thick  2=thin  3=shadow)")
axes[1].axis("off")
axes[2].imshow(rgb_display[rs:re, cs:ce])
axes[2].imshow(rgba[rs:re, cs:ce])
axes[2].set_title("Overlay")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 2. Batch processing

For many scenes, use `predict_from_load_func`. It processes each scene in turn, saves masks as GeoTIFFs alongside the inputs, and overlaps loading with inference automatically.

In [ ]:
scenes_dir = Path("/example/path/planetscope_scenes")
output_dir = Path("/example/path/output")

scene_paths = list(scenes_dir.glob("*.tif"))
print(f"Found {len(scene_paths)} scenes")

In [ ]:
load_func = partial(load_multiband, resample_res=10, band_order=ps_band_order)

mask_paths = predict_from_load_func(
    scene_paths=scene_paths,
    load_func=load_func,
    output_dir=output_dir,
)
print(f"Saved {len(mask_paths)} masks")